# Práctica 3: Procesamiento del lenguaje natural
# Inteligencia Artificial
# Grado en Ingeniería Informática - Ingeniería del Software
# Universidad de Sevilla

[NLTK](https://www.nltk.org) (_Natural Language Toolkit_) es una biblioteca de Python para trabajar con datos del lenguaje humano. Proporciona interfaces fáciles de usar para más de 50 corpus y recursos léxicos, junto con un conjunto de herramientas de procesamiento de texto para clasificación, tokenización, derivación, etiquetado, análisis y razonamiento semántico. Esta práctica presenta una introducción a esa biblioteca, mostrando cómo construir un filtro de correo electrónico no deseado y un sistema de texto predictivo.

En esta práctica también se hará uso de la biblioteca scikit-learn.

En primer lugar establecemos la semilla aleatoria inicial, para que el cuaderno sea reproducible.

In [2]:
import numpy as np
np.random.seed(357823)

## Filtro de correo electrónico no deseado

En la comunicación por correo electrónico se emplea el término _spam_ para designar a aquellos mensajes enviados de forma masiva y que no han sido solicitados por sus destinatarios. Los gestores de correo electrónico suelen incorporar la posibilidad de crear filtros de mensajes no deseados que permitan identificar estos de forma automática y actuar en consecuencia (usualmente moviéndolos de la bandeja de
entrada a una carpeta específica).

En esta primera parte de la práctica se va a construir un filtro de correo electrónico que, aplicando técnicas de aprendizaje automático, identifique lo mejor posible los mensajes no deseados. En concreto, se van a considerar los siguientes modelos predictivos:

* Modelo naive Bayes multinomial, usando el modelo de bolsa de palabras como modelo de lenguaje. Se probarán distintos valores del parámetro de suavizado.
* Modelo $k$NN, usando el modelo tf-idf como modelo de lenguaje. Se probarán distintos valores del número de vecinos.

Para entrenar los modelos y evaluar su rendimiento se usará [Enron-Spam](https://auebgr-my.sharepoint.com/:f:/g/personal/nlp_aueb_gr/Em7LRLHD3p9OqKOB3oK8BXoBZMANfisgs-jg0VLsZGQGWw?e=vJdUIF), que es un conjunto público de mensajes (en inglés) de correo electrónico, ya preclasificados en mensajes legítimos y no deseados. Este conjunto se encuentra en la carpeta `Filtro antispam/Enron-Spam/`, dividido en las subcarpetas `train/` (subconjunto de entrenamiento) y `test/` (subconjunto de prueba). En cada una de estas subcarpetas, los mensajes se encuentran organizados, a su vez, en subcarpetas `legítimo` y `no_deseado`.

Los mensajes se proporcionan en bruto, incluyendo las cabeceras. Para su lectura usaremos el paquete [email](https://docs.python.org/es/3/library/email.html) de la biblioteca estándar de Python. Por ejemplo, el contenido del mensaje no deseado `5` del subconjunto de entrenamiento se puede obtener con el siguiente código.

In [2]:
from email import parser
from email import policy

In [11]:
analizador_mensaje = parser.Parser(policy=policy.default)
with open('Filtro antispam/Enron-Spam/train/no_deseado/5') as fichero_mensaje:
    mensaje = analizador_mensaje.parse(fichero_mensaje)
    contenido_mensaje_5 = mensaje.get_content()
contenido_mensaje_5

"Hi, \nI have a special offer available for you at our casino.\n\n$20 to try our internet casino, no deposit is necessary!\nAt the casino software's cashier enter bonus code: FR93P\n\n$200 bonus on your first deposit!\nAt the casino software's cashier enter bonus code: FMJKU\n\nAllow us to show you our quality operation, fast payouts,\ngenerous bonuses, and super friendly around-the-clock\ncustomer support.\n\nClick here: http://bigbonus-casino.net \n\nBest regards,\nJamie Zawinsky\n\n\n\nNo thanks: http://bigbonus-casino.com/u/\n\n"

Antes de poder entrenar un modelo de aprendizaje automático capaz de discriminar los mensajes legítimos de los no deseados, es necesario vectorizar el contenido de los mensajes. Es decir, representar esos contenidos mediante vectores numéricos.

En esta práctica, la vectorización del contenido de los mensajes se va a llevar a cabo haciendo uso de un vocabulario fijo de términos que aparecen habitualmente en correos electrónicos no deseados. Ese vocabulario se ha extraído de [spam_words_api_lists](https://github.com/roumilb/spam_words_api_lists) y se encuentra en el fichero `Filtro antispam/términos_spam.txt`.

In [ ]:
with open('Filtro antispam/términos_spam.txt', 'r') as f:
    términos_spam = f.read()
términos_spam = términos_spam.split(sep='\n')

print(f"términos: {len(términos_spam)}")

términos: 494


El paquete `pprint` de la biblioteca estándar de Python facilita mostrar el vocabulario de una manera compacta.

In [17]:
from pprint import pprint

In [47]:
pprint(términos_spam, compact=True)
print(f"términos: {len(términos_spam)}")

['#1', '$$$', '$earn extra cash', '$save big money', '$save', '100% free',
 '100% satisfied', '100%', '4u', '50% off', 'accept credit cards', 'acceptance',
 'access', 'accordingly', 'act now!', 'act now', 'action', 'ad',
 'additional income', 'additional', 'addresses on cd', 'affordable',
 'all natural', 'all new', 'amazed', 'amazing stuff', 'amazing', 'americans',
 'apply now', 'apply online', 'as seen on', 'auto email removal',
 'avoid bankruptcy', 'avoid', 'bargain', 'be amazed', 'be your own boss',
 'being a member', 'beneficiary', 'best price', 'beverage', 'big bucks cash',
 'big bucks', 'bill 1618', 'billing address', 'billing', 'billion dollars',
 'billion', 'bonus', 'boss', 'brand new pager', 'bulk email', 'buy direct',
 'buy', 'buying judgments', 'cable converter', 'call free', 'call now', 'call',
 'calling creditors', "can't live without", 'cancel at any time', 'cancel',
 'cannot be combined with any other offer', 'can’t live without',
 'cards accepted', 'cash bonus', 'cash',

Obsérvese que el vocabulario contiene términos de hasta 9 palabras, algo que habrá que tener en cuenta al construir los vectorizadores del contenido de los mensajes.

In [19]:
max(len(término.split()) for término in términos_spam)

9

El primer vectorizador que vamos a construir es el modelo de bolsa de palabras, que representará el contenido de cada mensaje mediante **el vector del número de ocurrencias en el mensaje de cada término del vocabulario**. La clase `CountVectorizer` de scikit-learn implementa este modelo de lenguaje.

In [20]:
from sklearn.feature_extraction.text import CountVectorizer

Una vez creada una instancia de esta clase, sería posible, a través del método `fit`, pedirle que aprendiera el vocabulario de términos a partir de un corpus de entrenamiento. Nosotros, sin embargo, usamos un vocabulario fijo y, por tanto, hay que proporcionárselo explícitamente.

In [21]:
vectorizador = CountVectorizer(vocabulary=términos_spam)

Ahora basta aplicar el método `fit_transform` del vectorizador para obtener la representación como bolsa de palabras de la secuencia de contenidos de mensajes proporcionada. A la hora de mostrar esa representación, debe tenerse en cuenta que se obtiene en forma de matriz dispersa (_sparse matrix_).

**Nota**: aunque, como se ha comentado antes, el vocabulario es fijo, sigue siendo adecuado aplicar el método `fit`, ya que este realiza una serie de comprobaciones convenientes.

In [26]:
vectorizador.fit_transform([contenido_mensaje_5]).toarray()


array([[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 3, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 6, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 

El proceso de vectorización de una cadena consta de cuatro pasos:

1. En primer lugar, un preprocesador transforma la cadena a un formato adecuado. El preprocesamiento por defecto de CountVectorizer consiste en transformar la cadena a minúsculas.
2. A continuación, un tokenizador divide la cadena en una secuencia de tókenes. El tokenizador por defecto de CountVectorizer divide la cadena en tókenes constituidos por dos o más caracteres alfanuméricos.
3. Un analizador construye entonces los atributos que representan a la cadena. El analizador por defecto de CountVectorizer construye unigramas de tókenes.
4. Finalmente, se cuentan las ocurrencias de los atributos que sean términos del vocabulario.

### ¿Qué diferencia hay entre build_tokenizer() y build_analyzer()?

Diferencia clave:

**build_tokenizer()**
- Devuelve solo la función que separa texto en tokens.
- No genera n-gramas ni aplica todo el pipeline completo.
- Por eso en tu ejemplo lo llamas sobre el texto ya preprocesado (preprocesador(...)).

**build_analyzer()**
Devuelve la función completa de análisis de características para el vectorizador.
Incluye internamente:

- preprocesado (por ejemplo, minúsculas),
- tokenización,
- construcción de n-gramas según ngram_range,
- filtrados internos del vectorizador.
- Se le pasa el texto crudo directamente.

En tu caso concreto:

tokenizador(...) te devuelve palabras/tokens.
analizador(...) te devuelve las características finales que se van a contar (con tu configuración, desde 1-gramas hasta 9-gramas).
Regla práctica:

- Si quieres inspeccionar cómo se parte el texto: build_tokenizer.
- Si quieres ver exactamente qué usa CountVectorizer para crear el vector: build_analyzer.

In [27]:
preprocesador = vectorizador.build_preprocessor()
preprocesador(contenido_mensaje_5)

"hi, \ni have a special offer available for you at our casino.\n\n$20 to try our internet casino, no deposit is necessary!\nat the casino software's cashier enter bonus code: fr93p\n\n$200 bonus on your first deposit!\nat the casino software's cashier enter bonus code: fmjku\n\nallow us to show you our quality operation, fast payouts,\ngenerous bonuses, and super friendly around-the-clock\ncustomer support.\n\nclick here: http://bigbonus-casino.net \n\nbest regards,\njamie zawinsky\n\n\n\nno thanks: http://bigbonus-casino.com/u/\n\n"

In [33]:
tokenizador = vectorizador.build_tokenizer()
pprint(tokenizador(preprocesador(contenido_mensaje_5)),
       compact=True)


['hi', 'have', 'special', 'offer', 'available', 'for', 'you', 'at', 'our',
 'casino', '20', 'to', 'try', 'our', 'internet', 'casino', 'no', 'deposit',
 'is', 'necessary', 'at', 'the', 'casino', 'software', 'cashier', 'enter',
 'bonus', 'code', 'fr93p', '200', 'bonus', 'on', 'your', 'first', 'deposit',
 'at', 'the', 'casino', 'software', 'cashier', 'enter', 'bonus', 'code',
 'fmjku', 'allow', 'us', 'to', 'show', 'you', 'our', 'quality', 'operation',
 'fast', 'payouts', 'generous', 'bonuses', 'and', 'super', 'friendly', 'around',
 'the', 'clock', 'customer', 'support', 'click', 'here', 'http', 'bigbonus',
 'casino', 'net', 'best', 'regards', 'jamie', 'zawinsky', 'no', 'thanks',
 'http', 'bigbonus', 'casino', 'com']


In [34]:
analizador = vectorizador.build_analyzer()
pprint(analizador(contenido_mensaje_5),
       compact=True)

['hi', 'have', 'special', 'offer', 'available', 'for', 'you', 'at', 'our',
 'casino', '20', 'to', 'try', 'our', 'internet', 'casino', 'no', 'deposit',
 'is', 'necessary', 'at', 'the', 'casino', 'software', 'cashier', 'enter',
 'bonus', 'code', 'fr93p', '200', 'bonus', 'on', 'your', 'first', 'deposit',
 'at', 'the', 'casino', 'software', 'cashier', 'enter', 'bonus', 'code',
 'fmjku', 'allow', 'us', 'to', 'show', 'you', 'our', 'quality', 'operation',
 'fast', 'payouts', 'generous', 'bonuses', 'and', 'super', 'friendly', 'around',
 'the', 'clock', 'customer', 'support', 'click', 'here', 'http', 'bigbonus',
 'casino', 'net', 'best', 'regards', 'jamie', 'zawinsky', 'no', 'thanks',
 'http', 'bigbonus', 'casino', 'com']


Para nuestros propósitos, nos interesa usar un tokenizador más avanzado que el usado por defecto. En concreto, usaremos el tokenizador recomendado por NLTK, que divide cualquier texto en párrafos, frases y palabras como sigue: las líneas en blanco separan los párrafos; las frases se separan en palabras formadas bien por secuencias de caracteres alfanuméricos, bien por secuencias de caracteres no alfanuméricos ni espacios (que se descartan); los párrafos se separan en frases usando un modelo entrenado mediante aprendizaje no supervisado que se ha comprobado que funciona bien para muchos lenguajes europeos.

El tokenizador está implementado por la función `word_tokenize` del módulo `nltk.tokenize`, que usa el modelo de separación de párrafos en frases implementado en el módulo `nltk.tokenize.punkt`. En principio, la documentación recomienda entrenar este modelo con un corpus adecuado para la tarea que se pretende realizar. No obstante, NLTK proporciona modelo preentrenados para distintos idiomas, pero para poder usarlos es necesario descargar previamente sus parámetros.

In [35]:
# Los corpus, gramáticas, tokenizadores, etcétera que proporciona NLTK se descargan y buscan en ciertas carpetas por defecto.
# Para usar una carpeta distinta a ellas es necesario establecer la variable de entorno NLTK_DATA, con la prevención de hacerlo
# antes de cargar el paquete nltk

import os
os.environ['NLTK_DATA'] = '.'

In [37]:
from nltk import download

download('punkt_tab')

[nltk_data] Downloading package punkt_tab to ....
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


True

In [38]:
from nltk.tokenize import word_tokenize

Basta entonces indicarle al vectorizador que use el tokenizador proporcionado por NLTK que, además, por defecto usa el modelo de separación en frases preentrenado para el idioma inglés. También nos interesa que los atributos construidos por el analizador sean desde unigramas hasta 9-gramas.

In [39]:
vectorizador = CountVectorizer(vocabulary=términos_spam,
                               token_pattern=None,  # Anulamos el tokenizador por defecto basado en expresiones regulares
                               tokenizer=word_tokenize,  # Usamos el tokenizador proporcionado por NLTK
                               ngram_range=(1, 9))

In [40]:
tokenizador = vectorizador.build_tokenizer()
pprint(tokenizador(preprocesador(contenido_mensaje_5)),
       compact=True)

['hi', ',', 'i', 'have', 'a', 'special', 'offer', 'available', 'for', 'you',
 'at', 'our', 'casino', '.', '$', '20', 'to', 'try', 'our', 'internet',
 'casino', ',', 'no', 'deposit', 'is', 'necessary', '!', 'at', 'the', 'casino',
 'software', "'s", 'cashier', 'enter', 'bonus', 'code', ':', 'fr93p', '$',
 '200', 'bonus', 'on', 'your', 'first', 'deposit', '!', 'at', 'the', 'casino',
 'software', "'s", 'cashier', 'enter', 'bonus', 'code', ':', 'fmjku', 'allow',
 'us', 'to', 'show', 'you', 'our', 'quality', 'operation', ',', 'fast',
 'payouts', ',', 'generous', 'bonuses', ',', 'and', 'super', 'friendly',
 'around-the-clock', 'customer', 'support', '.', 'click', 'here', ':', 'http',
 ':', '//bigbonus-casino.net', 'best', 'regards', ',', 'jamie', 'zawinsky',
 'no', 'thanks', ':', 'http', ':', '//bigbonus-casino.com/u/']


In [41]:
analizador = vectorizador.build_analyzer()
pprint(analizador(contenido_mensaje_5),
       compact=True)

['hi', ',', 'i', 'have', 'a', 'special', 'offer', 'available', 'for', 'you',
 'at', 'our', 'casino', '.', '$', '20', 'to', 'try', 'our', 'internet',
 'casino', ',', 'no', 'deposit', 'is', 'necessary', '!', 'at', 'the', 'casino',
 'software', "'s", 'cashier', 'enter', 'bonus', 'code', ':', 'fr93p', '$',
 '200', 'bonus', 'on', 'your', 'first', 'deposit', '!', 'at', 'the', 'casino',
 'software', "'s", 'cashier', 'enter', 'bonus', 'code', ':', 'fmjku', 'allow',
 'us', 'to', 'show', 'you', 'our', 'quality', 'operation', ',', 'fast',
 'payouts', ',', 'generous', 'bonuses', ',', 'and', 'super', 'friendly',
 'around-the-clock', 'customer', 'support', '.', 'click', 'here', ':', 'http',
 ':', '//bigbonus-casino.net', 'best', 'regards', ',', 'jamie', 'zawinsky',
 'no', 'thanks', ':', 'http', ':', '//bigbonus-casino.com/u/', 'hi ,', ', i',
 'i have', 'have a', 'a special', 'special offer', 'offer available',
 'available for', 'for you', 'you at', 'at our', 'our casino', 'casino .',
 '. $', '$ 20',

In [48]:
vectorizador.fit_transform([contenido_mensaje_5]).toarray()[0]

array([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 3, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 4, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0,
       1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0,
       0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,

Estamos ya en condiciones de construir la representación como bolsa de palabras de todos los mensajes de entrenamiento.

**Nota**: el formato de algunos de los mensajes requiere de un código elaborado para su adecuada lectura, por lo que simplemente haremos uso de una estructura `try...except` para descartar aquellos mensajes que produzcan error al tratar de leerlos con nuestro código simple.

Para una mayor flexibilidad en la gestión de las rutas de los ficheros, nos apoyaremos en el paquete [pathlib](https://docs.python.org/es/3/library/pathlib.html) de la biblioteca estándar de Python.

In [49]:
from pathlib import Path

In [50]:
carpeta_Enron_Spam = Path('Filtro antispam/Enron-Spam/')
carpeta_entrenamiento = carpeta_Enron_Spam / 'train'

In [51]:
contenidos_mensajes_entrenamiento = []
clases_mensajes_entrenamiento = []

# Leemos los mensajes legítimos (clase 0)
for ruta_mensaje in (carpeta_entrenamiento / 'legítimo').iterdir():
    with open(ruta_mensaje, 'r') as fichero_mensaje:
        try:
            mensaje = analizador_mensaje.parse(fichero_mensaje)
            contenidos_mensajes_entrenamiento.append(mensaje.get_content())
            clases_mensajes_entrenamiento.append(0)
        except (KeyError, UnicodeDecodeError, LookupError):
            pass

# Leemos los mensajes no deseados (clase 1)
for ruta_mensaje in (carpeta_entrenamiento / 'no_deseado').iterdir():
    with open(ruta_mensaje, 'r') as fichero_mensaje:
        try:
            mensaje = analizador_mensaje.parse(fichero_mensaje)
            contenidos_mensajes_entrenamiento.append(mensaje.get_content())
            clases_mensajes_entrenamiento.append(1)
        except (KeyError, UnicodeDecodeError, LookupError):
            pass

In [52]:
bolsa_de_palabras_entrenamiento = vectorizador.transform(
    contenidos_mensajes_entrenamiento
)
bolsa_de_palabras_entrenamiento

<Compressed Sparse Row sparse matrix of dtype 'int64'
	with 84608 stored elements and shape (20991, 494)>

Ahora haremos una búsqueda en rejilla para determinar el mejor valor de suavizado para el modelo naive Bayes multinomial. Como nuestro objetivo es identificar los mensajes no deseados, elegimos la sensibilidad como medida de rendimiento.

In [53]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.model_selection import GridSearchCV

In [26]:
filtro_NB = MultinomialNB()
búsqueda_en_rejilla = GridSearchCV(
    filtro_NB,
    {'alpha': range(1, 5)},
    scoring='recall',
    cv=5
)
búsqueda_en_rejilla.fit(bolsa_de_palabras_entrenamiento,
                        clases_mensajes_entrenamiento)

GridSearchCV(cv=5, estimator=MultinomialNB(), param_grid={'alpha': range(1, 5)},
             scoring='recall')

In [27]:
búsqueda_en_rejilla.best_params_

{'alpha': 1}

In [28]:
búsqueda_en_rejilla.best_score_

np.float64(0.616353887399464)

Para construir un filtro de correo electrónico no deseado usando el modelo tf-idf como modelo de lenguaje y el modelo $k$NN como modelo predictivo, basta replicar convenientemente los pasos realizados anteriormente.

In [54]:
from sklearn.feature_extraction.text import TfidfVectorizer

In [56]:
vectorizador = TfidfVectorizer(vocabulary=términos_spam,
                               token_pattern=None,
                               tokenizer=word_tokenize,
                               ngram_range=(1, 9))
# Calculamos la frecuencia documental inversa a partir de todos los
# mensajes de entrenamiento
vectorizador.fit(contenidos_mensajes_entrenamiento)
# Construimos la representación tf-idf del mensaje no deseado 5
vectorizador.transform([contenido_mensaje_5]).toarray()

array([[0.        , 0.        , 0.        , 0.        , 0.        ,
        0.        , 0.        , 0.        , 0.        , 0.        ,
        0.        , 0.        , 0.        , 0.        , 0.        ,
        0.        , 0.        , 0.        , 0.        , 0.        ,
        0.        , 0.        , 0.        , 0.        , 0.        ,
        0.        , 0.        , 0.        , 0.        , 0.        ,
        0.        , 0.        , 0.        , 0.        , 0.        ,
        0.        , 0.        , 0.        , 0.        , 0.        ,
        0.        , 0.        , 0.        , 0.        , 0.        ,
        0.        , 0.        , 0.        , 0.48457398, 0.        ,
        0.        , 0.        , 0.        , 0.        , 0.        ,
        0.        , 0.        , 0.        , 0.        , 0.        ,
        0.        , 0.        , 0.        , 0.        , 0.        ,
        0.        , 0.        , 0.        , 0.        , 0.8362666 ,
        0.        , 0.        , 0.        , 0.  

Sin embargo, en este caso existe la dificultad de que la frecuencia documental inversa debe calcularse únicamente a partir del corpus de entrenamiento. Esto quiere decir que la búsqueda en rejilla debe realizar tanto la vectorización de los mensajes como la construcción del modelo predictivo.

Es necesario, entonces, el uso de las tuberías de scikit-learn. Además, por eficiencia, se usará la clase `TfidfTransformer` para obtener directamente las frecuencias documentales inversas a partir de la bolsa de palabras ya construida.

In [57]:
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfTransformer
from sklearn.neighbors import KNeighborsClassifier

In [58]:
tubería_filtro_kNN = Pipeline([
    ('vectorizador', TfidfTransformer()),
    ('filtro_kNN', KNeighborsClassifier(metric='cosine'))
])

In [59]:
búsqueda_en_rejilla = GridSearchCV(
    tubería_filtro_kNN,
    {'filtro_kNN__n_neighbors': range(1, 6, 2)},
    scoring='recall',
    cv=5
)
búsqueda_en_rejilla.fit(bolsa_de_palabras_entrenamiento,
                        clases_mensajes_entrenamiento)

,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",Pipeline(step...c='cosine'))])
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'filtro_kNN__n_neighbors': range(1, 6, 2)}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion ` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",'recall'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",None
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide ` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",5
,"verbose verbose: intControls the verbosity: the higher, the more messages.- >1 : the computation time for each fold and parameter candidate is displayed;- >2 : the score is also displayed;- >3 : the fold

In [60]:
búsqueda_en_rejilla.best_params_

{'filtro_kNN__n_neighbors': 1}

In [61]:
búsqueda_en_rejilla.best_score_

np.float64(0.8348759221998658)

La conclusión es, por tanto, que de todos los filtros considerados, el que mejor identifica los mensajes no deseados es el modelo $k$NN con el valor 1 para el número de vecinos.

Procedemos a entrenar ese modelo sobre todo el corpus de entrenamiento y a evaluar su rendimiento sobre el corpus de prueba.

In [62]:
filtro_seleccionado = Pipeline([
    ('vectorizador', TfidfVectorizer(vocabulary=términos_spam,
                                     token_pattern=None,
                                     tokenizer=word_tokenize,
                                     ngram_range=(1, 9))),
    ('filtro_kNN', KNeighborsClassifier(metric='cosine',
                                        n_neighbors=1))
])

In [63]:
filtro_seleccionado.fit(contenidos_mensajes_entrenamiento,
                        clases_mensajes_entrenamiento)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('vectorizador', ...), ('filtro_kNN', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"input input: {'filename', 'file', 'content'}, default='content'- If `'filename'`, the sequence passed as an argument to fit is expected to be a list of filenames that need reading to fetch the raw content to analyze.- If `'file'`, the sequence items must have a 'read' method (file-like object) that is called to fetch the bytes in memory.- If `'content'`, the input is expected to be a sequence of items that can be of type string or byte.",'content'
,"encoding encoding: str, default='utf-8'If bytes or files are given to analyze, this encoding is used todecode.",'utf-8'
,"decode_error decode_error: {'strict', 'ignore', 'replace'}, default='strict'Instruction on what to do if a byte sequence is given to analyze thatcontains characters not of the given `encoding`. By default, it is'strict', meaning that a UnicodeDecodeError will be raised. Othervalues are 'ignore' and 'replace'.",'strict'
,"strip_accents strip_accents: {'ascii', 'unicode'} or callable, default=NoneRemove accents and perform other character normalizationduring the preprocessing step.'ascii' is a fast method that only works on characters that havea direct ASCII mapping.'unicode' is a slightly slower method that works on any characters.None (default) means no character normalization is performed.Both 'ascii' and 'unicode' use NFKD normalization from:func:`unicodedata.normalize`.",None
,"lowercase lowercase: bool, default=TrueConvert all characters to lowercase before tokenizing.",True
,"preprocessor preprocessor: callable, default=NoneOverride the preprocessing (string transformation) stage whilepreserving the tokenizing and n-grams generation steps.Only applies if ``analyzer`` is not callable.",None
,"tokenizer tokenizer: callable, default=NoneOverride the string tokenization step while preserving thepreprocessing and n-grams generation steps.Only applies if ``analyzer == 'word'``.",<function wor...x7125d85798a0>


In [64]:
carpeta_prueba = carpeta_Enron_Spam / 'test'

contenidos_mensajes_prueba = []
clases_mensajes_prueba = []

# Leemos los mensajes legítimos (clase 0)
for ruta_mensaje in (carpeta_prueba / 'legítimo').iterdir():
    with open(ruta_mensaje, 'r') as fichero_mensaje:
        try:
            mensaje = analizador_mensaje.parse(fichero_mensaje)
            contenidos_mensajes_prueba.append(mensaje.get_content())
            clases_mensajes_prueba.append(0)
        except (KeyError, UnicodeDecodeError, LookupError):
            pass

# Leemos los mensajes no deseados (clase 1)
for ruta_mensaje in (carpeta_prueba / 'no_deseado').iterdir():
    with open(ruta_mensaje, 'r') as fichero_mensaje:
        try:
            mensaje = analizador_mensaje.parse(fichero_mensaje)
            contenidos_mensajes_prueba.append(mensaje.get_content())
            clases_mensajes_prueba.append(1)
        except (KeyError, UnicodeDecodeError, LookupError):
            pass

In [65]:
from sklearn.metrics import recall_score

In [66]:
predicciones_mensajes_prueba = filtro_seleccionado.predict(
    contenidos_mensajes_prueba)
recall_score(clases_mensajes_prueba, predicciones_mensajes_prueba)

0.8380849919311458

## Sistema de texto predictivo

Los sistemas de texto predictivo son una tecnología de entrada de texto diseñada para dispositivos móviles. Esta tecnología permite formar palabras presionando una zona de la pantalla asociada a un grupo de letras. La aplicación principal de esta tecnología es simplificar la escritura de mensajes de texto.

Aunque esta tecnología se utilizó inicialmente para facilitar la escritura de mensajes en teléfonos móviles con teclado numérico, la aparición del smartphone con teclado ampliado provocó que cambiase el tipo de aplicaciones a las que se aplica. Actualmente se utiliza tanto para facilitar la escritura en teclados ampliados (samsung swype) como para sugerir nuevas entradas de texto (escritura inteligente). También es notable su uso en dispositivos móviles más pequeños como los Smart Watch.

Los sistemas de texto predicen la palabra que queremos escribir a partir de las pulsaciones realizadas. Para ello han efectuado previamente un análisis estadístico de un corpus de textos de referencia, determinando las probabilidades de las correspondencias entre distintas secuencias de pulsaciones y posibles palabras.

Estos sistemas suelen mostrar la palabra que corresponde con mayor probabilidad a la combinación de pulsaciones realizada por el usuario, actualizándose esta palabra a medida que el usuario realiza nuevas pulsaciones. Además, es habitual ofrecer al usuario la posibilidad de requerir otras posibilidades, aparte de aceptar la palabra propuesta por el sistema.

La efectividad de un sistema de prediccion de texto dependerá de varios factores:

* La calidad del corpus.
* El modelo de lenguaje obtenido a partir del análisis estadístico del corpus.

En esta segunda parte de la práctica se va utilizar la biblioteca NLTK para construir un sistema de predicción de texto en español basado en modelos de $n$-gramas.

En primer lugar se debe cargar un corpus de textos en español que nos permita entrenar el modelo. Por ejemplo, el [corpus InfoLibros](https://zenodo.org/records/7313105) es un corpus de 218 millones de tókenes de narrativas españolas extraídas de libros gratuitos recopilados por el proyecto abierto [Infolibros.org](http://infolibros.org/). Por motivos de eficiencia computacional, nosotros usaremos únicamente una parte de ese corpus, guardado en el fichero `Texto predictivo/corpus_InfoLibros_parcial.txt`.

NLTK proporciona diferentes clases que permiten leer corpus de muy diversos tipos. Entre ellas se encuentra la clase `PlaintextCorpusReader` para leer corpus proporcionados en ficheros de texto plano, como es el caso que nos ocupa. Para identificar las frases contenidas en el corpus, usaremos el modelo proporcionado por NLTK para el español, creando para ello una instancia adecuada de la clase `PunktTokenizer`. Mantendremos la separación por defecto de cada una de esas frases en palabras que sean secuencias solo de caracteres alfanuméricos o solo de caracteres no alfanuméricos, aunque ese comportamiento también es configurable.

In [67]:
# Nos aseguramos de haber descargado el tokenizador

from nltk import download

download('punkt_tab')

[nltk_data] Downloading package punkt_tab to ....
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [68]:
from nltk.corpus.reader.plaintext import PlaintextCorpusReader
from nltk.tokenize.punkt import PunktTokenizer

In [69]:
corpus_InfoLibros = PlaintextCorpusReader(
    root='Texto predictivo',
    fileids=['corpus_InfoLibros_parcial.txt'],
    encoding='utf8',
    sent_tokenizer=PunktTokenizer(lang='spanish')
)

Los métodos `paras`, `sents` y `words` proporcionan, respectivamente, los párrafos, frases y palabras identificados en el corpus.

In [70]:
for para in corpus_InfoLibros.paras()[:5]:
    pprint(para[:3] + [['...']], compact=True)
    print()

[['Venga', 'usted', 'aquí', ',', 'querida', 'Elena', '-', 'dijo', 'Ana',
  'Pavlovna', 'a', 'la', 'bella', 'Princesa', ',', 'que', ',', 'sentada', 'un',
  'poco', 'más', 'lejos', ',', 'formaba', 'el', 'centro', 'del', 'otro',
  'grupo', '.'],
 ['La', 'princesa', 'Elena', 'sonrió', 'y', 'se', 'levantó', 'con', 'la',
  'misma', 'invariable', 'sonrisa', 'de', 'mujer', 'absolutamente', 'hermosa',
  'con', 'que', 'había', 'entrado', 'en', 'el', 'salón', '.'],
 ['Con', 'el', 'ligero', 'rumor', 'de', 'su', 'leve', 'vestido', 'de', 'baile',
  'con', 'adornos', 'de', 'felpa', ',', 'deslumbradora', 'por', 'la',
  'blancura', 'de', 'sus', 'hombros', 'y', 'el', 'esplendor', 'de', 'sus',
  'cabellos', 'y', 'de', 'sus', 'diamantes', ',', 'cruzó', 'entre', 'los',
  'hombres', ',', 'que', 'le', 'abrieron', 'paso', ',', 'rígida', ',', 'sin',
  'ver', 'a', 'nadie', ',', 'pero', 'sonriendo', 'a', 'todos', 'como', 'si',
  'concediese', 'a', 'cada', 'uno', 'el', 'derecho', 'de', 'admirar', 'la',
  'belleza

In [71]:
pprint(corpus_InfoLibros.sents()[:5], compact=True)

[['Venga', 'usted', 'aquí', ',', 'querida', 'Elena', '-', 'dijo', 'Ana',
  'Pavlovna', 'a', 'la', 'bella', 'Princesa', ',', 'que', ',', 'sentada', 'un',
  'poco', 'más', 'lejos', ',', 'formaba', 'el', 'centro', 'del', 'otro',
  'grupo', '.'],
 ['La', 'princesa', 'Elena', 'sonrió', 'y', 'se', 'levantó', 'con', 'la',
  'misma', 'invariable', 'sonrisa', 'de', 'mujer', 'absolutamente', 'hermosa',
  'con', 'que', 'había', 'entrado', 'en', 'el', 'salón', '.'],
 ['Con', 'el', 'ligero', 'rumor', 'de', 'su', 'leve', 'vestido', 'de', 'baile',
  'con', 'adornos', 'de', 'felpa', ',', 'deslumbradora', 'por', 'la',
  'blancura', 'de', 'sus', 'hombros', 'y', 'el', 'esplendor', 'de', 'sus',
  'cabellos', 'y', 'de', 'sus', 'diamantes', ',', 'cruzó', 'entre', 'los',
  'hombres', ',', 'que', 'le', 'abrieron', 'paso', ',', 'rígida', ',', 'sin',
  'ver', 'a', 'nadie', ',', 'pero', 'sonriendo', 'a', 'todos', 'como', 'si',
  'concediese', 'a', 'cada', 'uno', 'el', 'derecho', 'de', 'admirar', 'la',
  'belleza

In [72]:
pprint(corpus_InfoLibros.words()[:50], compact=True)

['Venga', 'usted', 'aquí', ',', 'querida', 'Elena', '-', 'dijo', 'Ana',
 'Pavlovna', 'a', 'la', 'bella', 'Princesa', ',', 'que', ',', 'sentada', 'un',
 'poco', 'más', 'lejos', ',', 'formaba', 'el', 'centro', 'del', 'otro', 'grupo',
 '.', 'La', 'princesa', 'Elena', 'sonrió', 'y', 'se', 'levantó', 'con', 'la',
 'misma', 'invariable', 'sonrisa', 'de', 'mujer', 'absolutamente', 'hermosa',
 'con', 'que', 'había', 'entrado']


Siguiendo el procedimiento habitual en aprendizaje automático, dividimos el corpus en un subcorpus de entrenamiento y un subcorpus de prueba. Sin embargo, debido al tamaño del corpus, una división aleatoria resulta muy costosa computacionalmente. Es por ello que para el corpus de entrenamiento seleccionaremos el 80 % de las frases iniciales del corpus, dejando para el corpus de prueba el 20 % de las frases finales.

In [73]:
total_frases = len(corpus_InfoLibros.sents())
total_frases

1008664

In [74]:
total_frases_entrenamiento = int(total_frases * .8)
total_frases_entrenamiento

806931

In [75]:
corpus_entrenamiento = corpus_InfoLibros.sents()[:total_frases_entrenamiento]
corpus_prueba = corpus_InfoLibros.sents()[total_frases_entrenamiento:]

El primer paso en la construcción del sistema de texto predictivo consiste en identificar el vocabulario de todas las palabras que aparecen en el corpus de entrenamiento. Para ello se hará uso de la clase `Vocabulary`, que proporciona una estructura de tipo diccionario que guarda el número de ocurrencias de cada palabra identificada. Esto permite establecer un número mínimo de ocurrencias para que una palabra se considere parte del vocabulario.

In [76]:
from nltk.lm.vocabulary import Vocabulary
# flatten es una utilidad que permite aplanar listas de listas
from nltk.lm.preprocessing import flatten

In [77]:
vocabulario_palabras = Vocabulary(
    flatten(corpus_entrenamiento),  # lista de todas las palabras
    unk_cutoff=50  # mínimo número de ocurrencias
)

El vocabulario incluye por defecto el término `<UNK>` para representar los términos desconocidos que no aparecen en el corpus de entrenamiento.

In [78]:
# Hola aparece en el corpus de entrenamiento y, por tanto,
# pertenece al vocabulario
vocabulario_palabras.lookup('Hola')

'Hola'

In [79]:
# hola no aparece en el corpus de entrenamiento y, por tanto,
# es un término desconocido
vocabulario_palabras.lookup('hola')

'<UNK>'

Por otra parte, los símbolos especiales de inicio y fin de secuencia que es necesario considerar en los modelos de $n$-gramas hay que incluirlos explícitamente en el vocabulario y, además, en este caso con frecuencia 50, para que sean considerados parte del vocabulario.

In [80]:
inicio_frase = '<s>'
fin_frase = '</s>'
vocabulario_palabras.update({inicio_frase: 50, fin_frase: 50})

La siguiente función facilitará delimitar adecuadamente, en función de los n-gramas pretendidos, las frases con esos símbolos especiales.

In [81]:
def delimita_frase(frase, n):
    return ['<s>'] * (n - 1) + frase + ['</s>']

In [82]:
primera_frase_entrenamiento = corpus_entrenamiento[0]
pprint(delimita_frase(primera_frase_entrenamiento, 1),
       compact=True)
pprint(delimita_frase(primera_frase_entrenamiento, 2),
       compact=True)
pprint(delimita_frase(primera_frase_entrenamiento, 3),
       compact=True)

['Venga', 'usted', 'aquí', ',', 'querida', 'Elena', '-', 'dijo', 'Ana',
 'Pavlovna', 'a', 'la', 'bella', 'Princesa', ',', 'que', ',', 'sentada', 'un',
 'poco', 'más', 'lejos', ',', 'formaba', 'el', 'centro', 'del', 'otro', 'grupo',
 '.', '</s>']
['<s>', 'Venga', 'usted', 'aquí', ',', 'querida', 'Elena', '-', 'dijo', 'Ana',
 'Pavlovna', 'a', 'la', 'bella', 'Princesa', ',', 'que', ',', 'sentada', 'un',
 'poco', 'más', 'lejos', ',', 'formaba', 'el', 'centro', 'del', 'otro', 'grupo',
 '.', '</s>']
['<s>', '<s>', 'Venga', 'usted', 'aquí', ',', 'querida', 'Elena', '-', 'dijo',
 'Ana', 'Pavlovna', 'a', 'la', 'bella', 'Princesa', ',', 'que', ',', 'sentada',
 'un', 'poco', 'más', 'lejos', ',', 'formaba', 'el', 'centro', 'del', 'otro',
 'grupo', '.', '</s>']


Para obtener los $n$-gramas de una frase basta aplicar la función `ngrams` de NLTK, o específicamente las funciones `bigrams` y `trigrams` para los bigramas y trigramas.

In [83]:
from nltk.util import ngrams, bigrams, trigrams

In [84]:
list(ngrams(delimita_frase(primera_frase_entrenamiento, 1), n=1))

[('Venga',),
 ('usted',),
 ('aquí',),
 (',',),
 ('querida',),
 ('Elena',),
 ('-',),
 ('dijo',),
 ('Ana',),
 ('Pavlovna',),
 ('a',),
 ('la',),
 ('bella',),
 ('Princesa',),
 (',',),
 ('que',),
 (',',),
 ('sentada',),
 ('un',),
 ('poco',),
 ('más',),
 ('lejos',),
 (',',),
 ('formaba',),
 ('el',),
 ('centro',),
 ('del',),
 ('otro',),
 ('grupo',),
 ('.',),
 ('</s>',)]

In [85]:
list(bigrams(delimita_frase(primera_frase_entrenamiento, 2)))

[('<s>', 'Venga'),
 ('Venga', 'usted'),
 ('usted', 'aquí'),
 ('aquí', ','),
 (',', 'querida'),
 ('querida', 'Elena'),
 ('Elena', '-'),
 ('-', 'dijo'),
 ('dijo', 'Ana'),
 ('Ana', 'Pavlovna'),
 ('Pavlovna', 'a'),
 ('a', 'la'),
 ('la', 'bella'),
 ('bella', 'Princesa'),
 ('Princesa', ','),
 (',', 'que'),
 ('que', ','),
 (',', 'sentada'),
 ('sentada', 'un'),
 ('un', 'poco'),
 ('poco', 'más'),
 ('más', 'lejos'),
 ('lejos', ','),
 (',', 'formaba'),
 ('formaba', 'el'),
 ('el', 'centro'),
 ('centro', 'del'),
 ('del', 'otro'),
 ('otro', 'grupo'),
 ('grupo', '.'),
 ('.', '</s>')]

In [86]:
list(trigrams(delimita_frase(primera_frase_entrenamiento, 3)))

[('<s>', '<s>', 'Venga'),
 ('<s>', 'Venga', 'usted'),
 ('Venga', 'usted', 'aquí'),
 ('usted', 'aquí', ','),
 ('aquí', ',', 'querida'),
 (',', 'querida', 'Elena'),
 ('querida', 'Elena', '-'),
 ('Elena', '-', 'dijo'),
 ('-', 'dijo', 'Ana'),
 ('dijo', 'Ana', 'Pavlovna'),
 ('Ana', 'Pavlovna', 'a'),
 ('Pavlovna', 'a', 'la'),
 ('a', 'la', 'bella'),
 ('la', 'bella', 'Princesa'),
 ('bella', 'Princesa', ','),
 ('Princesa', ',', 'que'),
 (',', 'que', ','),
 ('que', ',', 'sentada'),
 (',', 'sentada', 'un'),
 ('sentada', 'un', 'poco'),
 ('un', 'poco', 'más'),
 ('poco', 'más', 'lejos'),
 ('más', 'lejos', ','),
 ('lejos', ',', 'formaba'),
 (',', 'formaba', 'el'),
 ('formaba', 'el', 'centro'),
 ('el', 'centro', 'del'),
 ('centro', 'del', 'otro'),
 ('del', 'otro', 'grupo'),
 ('otro', 'grupo', '.'),
 ('grupo', '.', '</s>')]

La clase `MLE` implementa los modelos de $n$-gramas de máxima verosimilitud y la clase `Laplace` los modelos análogos que, adicionalmente, incorporan un suavizado de Laplace. Para crear instancias de esas clases hay que proporcionar el orden de los $n$-gramas y el vocabulario de términos. Una vez creadas, se entrenan aplicando el método `fit` a la secuencia de las secuencias de $n$-gramas de cada frase de entrenamiento y se evalua aplicando el método `perplexity` a la secuencia de los $n$-gramas de todas las frases de prueba.

In [87]:
from nltk.lm import MLE, Laplace

In [88]:
modelo_unigrama_MLE = MLE(1, vocabulary=vocabulario_palabras)
modelo_unigrama_MLE.fit(ngrams(delimita_frase(frase, 1), n=1)
                        for frase in corpus_entrenamiento)
modelo_unigrama_MLE.perplexity(flatten(ngrams(delimita_frase(frase, 1), n=1)
                                       for frase in corpus_prueba))

569.4380946356334

In [89]:
modelo_unigrama_Laplace = Laplace(1, vocabulary=vocabulario_palabras)
modelo_unigrama_Laplace.fit(ngrams(delimita_frase(frase, 1), n=1)
                            for frase in corpus_entrenamiento)
modelo_unigrama_Laplace.perplexity(flatten(ngrams(delimita_frase(frase, 1), n=1)
                                           for frase in corpus_prueba))

569.4578979334117

In [90]:
modelo_bigrama_MLE = MLE(2, vocabulary=vocabulario_palabras)
modelo_bigrama_MLE.fit(bigrams(delimita_frase(frase, 2))
                       for frase in corpus_entrenamiento)
modelo_bigrama_MLE.perplexity(flatten(bigrams(delimita_frase(frase, 2))
                                      for frase in corpus_prueba))

inf

La perplejidad no tiene una cota superior absoluta. Se calcula como el inverso de la probabilidad asignada por el modelo al corpus de prueba, normalizado por la cantidad de términos que este contiene.

- Puesto que el modelo puede asignar a una secuencia una probabilidad arbitrariamente pequeña (muy cercana a 0), el valor inverso puede crecer indefinidamente hacia el infinito
- De hecho, si el modelo se encuentra con una secuencia a la que le asigna una probabilidad de 0 (por ejemplo, si no se han aplicado técnicas de suavizado ante palabras desconocidas), la perplejidad se dispararía matemáticamente al **infinito**.


#### Se interpreta siguiendo esta regla principal:
En cuanto a su interpretación, la perplejidad evalúa el nivel de "sorpresa" o "duda" de un modelo de lenguaje al enfrentarse a un texto real nuevo. 

- Cuanto menor sea el valor de la perplejidad, mejor será el modelo. Un valor bajo (cuyo límite inferior ideal roza el 1) significa que el modelo ha asignado una probabilidad alta a la secuencia real, es decir, el texto encaja perfectamente con lo que el modelo esperaba leer.
- Una perplejidad muy alta indica que el modelo está sumamente "sorprendido" por la secuencia, ya que la considera altamente improbable basándose en lo que aprendió en su entrenamiento. Como vimos en los ejercicios anteriores, si una frase está compuesta puramente por elementos "raros" o poco frecuentes, la duda del modelo se dispara y la perplejidad alcanza valores máximos correspondientes a la inversa de esa probabilidad mínima.

In [91]:
modelo_bigrama_Laplace = Laplace(2, vocabulary=vocabulario_palabras)
modelo_bigrama_Laplace.fit(bigrams(delimita_frase(frase, 2))
                           for frase in corpus_entrenamiento)
modelo_bigrama_Laplace.perplexity(flatten(bigrams(delimita_frase(frase, 2))
                                          for frase in corpus_prueba))

366.2692890116132

In [66]:
modelo_trigrama_MLE = MLE(3, vocabulary=vocabulario_palabras)
modelo_trigrama_MLE.fit(trigrams(delimita_frase(frase, 3))
                        for frase in corpus_entrenamiento)
modelo_trigrama_MLE.perplexity(flatten(trigrams(delimita_frase(frase, 3))
                                       for frase in corpus_prueba))

inf

In [67]:
modelo_trigrama_Laplace = Laplace(3, vocabulary=vocabulario_palabras)
modelo_trigrama_Laplace.fit(trigrams(delimita_frase(frase, 3))
                            for frase in corpus_entrenamiento)
modelo_trigrama_Laplace.perplexity(flatten(trigrams(delimita_frase(frase, 3))
                                           for frase in corpus_prueba))

2138.444556966786

El mejor modelo parece ser, por tanto, el modelo bigrama con suavizado de Laplace, ya que es el que _menos perplejo se queda_ ante el corpus de prueba.

La idea es, entonces, usar estos modelos para predecir, a partir de las palabras anteriores y de las letras de la palabra ya escritas, qué palabra se pretende escribir.

In [92]:
def predice_palabras(prefijo, contexto, numero_palabras, modelo):
    def puntua_palabra(palabra):
        return modelo.score(palabra, contexto)  # contexto debe ser una tupla
    palabras_candidatas = filter(lambda palabra: palabra.startswith(prefijo),
                                 vocabulario_palabras)
    palabras_candidatas_ordenadas = sorted(palabras_candidatas,
                                           key=puntua_palabra,
                                           reverse=True)
    return palabras_candidatas_ordenadas[:numero_palabras]

Por ejemplo, si se ha escrito _Procesamiento del lenguaje nat_ podríamos usar el modelo bigrama con suavizado de Laplace (luego el contexto sería una tupla con la palabra _lenguaje_) para predecir las cinco palabras más probables que se estarían queriendo escribir.

In [93]:
predice_palabras('nat', ('lenguaje',), 5, modelo_bigrama_Laplace)

['natural', 'naturalmente', 'nata', 'naturales', 'naturaleza']